In [ ]:
# Import the analyzer and necessary libraries
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from visualize_prompt_consistency import PromptConsistencyAnalyzer

# For inline plotting
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully")

## 1. Load Data

Load all `other_analysis` files and parse within-run and cross-run metrics.

In [ ]:
# Initialize analyzer
base_path = Path.cwd()
analyzer = PromptConsistencyAnalyzer(base_path)

# Load all data
print("Loading data from all disciplines...")
data = analyzer.load_all_files()

print(f"\n{'='*70}")
print(f"Total topics analyzed: {len(data)}")
print(f"Topics with within-run problems: {data['has_within_run_problems'].sum()} ({data['has_within_run_problems'].sum()/len(data)*100:.1f}%)")
print(f"Average cross-run consistency: {data['cross_run_consistency_pct'].mean():.1f}%")
print(f"{'='*70}")

## 2. Data Overview

Examine the dataset structure and key metrics.

In [ ]:
# Display first few rows
print("Sample Data (first 10 topics):")
display(data[['discipline', 'topic', 'has_within_run_problems', 'within_run_duplicates', 
              'cross_run_consistency_pct', 'avg_questions_per_run']].head(10))

print("\nSummary Statistics by Discipline:")
summary = data.groupby('discipline').agg({
    'topic': 'count',
    'has_within_run_problems': 'sum',
    'within_run_duplicates': 'sum',
    'within_run_similar': 'sum',
    'cross_run_consistency_pct': ['mean', 'std'],
    'avg_questions_per_run': 'mean'
}).round(2)
summary.columns = ['Topics', 'With Problems', 'Total Within Duplicates', 'Total Within Similar',
                   'Avg Consistency %', 'Std Consistency %', 'Avg Q/Run']
display(summary)

## 3. Within-Run Problems Analysis

**Within-run problems** occur when a single prompt execution generates duplicate or very similar questions. This is problematic because the model should generate distinct questions in one run.

### Key Findings:

In [ ]:
from IPython.display import Image
Image('within_run_problems.png', width=1000)

In [ ]:
# Detailed analysis of within-run problems
print("Topics with Within-Run Problems:")
problem_topics = data[data['has_within_run_problems']].sort_values('within_run_duplicates', ascending=False)
display(problem_topics[['discipline', 'topic', 'within_run_duplicates', 'within_run_similar', 
                        'runs_with_duplicates', 'runs_with_similar']].head(15))

print(f"\n📊 Key Insights:")
print(f"  - {len(problem_topics)} out of {len(data)} topics have within-run issues ({len(problem_topics)/len(data)*100:.1f}%)")
print(f"  - Total within-run duplicates: {problem_topics['within_run_duplicates'].sum()}")
print(f"  - Total within-run similar: {problem_topics['within_run_similar'].sum()}")

# By discipline
print(f"\n📈 By Discipline:")
for disc in data['discipline'].unique():
    disc_data = data[data['discipline'] == disc]
    prob_count = disc_data['has_within_run_problems'].sum()
    print(f"  {disc}: {prob_count}/{len(disc_data)} topics ({prob_count/len(disc_data)*100:.1f}%)")

## 4. Cross-Run Consistency Analysis

**Cross-run consistency** measures how similar the generated questions are when running the same prompt multiple times. 

- **High consistency** (many duplicates/similar across runs) = Prompts produce predictable, repeatable questions
- **Low consistency** (few matches across runs) = Prompts produce varied, diverse questions

### Consistency Distribution:

In [ ]:
Image('cross_run_consistency.png', width=1000)

In [ ]:
# Analyze consistency patterns
print("Most Consistent Topics (highest similarity across runs):")
most_consistent = data.nlargest(10, 'cross_run_consistency_pct')
display(most_consistent[['discipline', 'topic', 'cross_run_consistency_pct', 
                         'cross_run_duplicates', 'cross_run_similar']].head(10))

print("\nLeast Consistent Topics (lowest similarity across runs):")
least_consistent = data.nsmallest(10, 'cross_run_consistency_pct')
display(least_consistent[['discipline', 'topic', 'cross_run_consistency_pct',
                          'cross_run_duplicates', 'cross_run_similar']].head(10))

print(f"\n📊 Consistency Statistics:")
print(f"  - Mean consistency: {data['cross_run_consistency_pct'].mean():.1f}%")
print(f"  - Median consistency: {data['cross_run_consistency_pct'].median():.1f}%")
print(f"  - Standard deviation: {data['cross_run_consistency_pct'].std():.1f}%")

## 5. Discipline Comparison

Compare prompt behavior across different disciplines.

In [ ]:
Image('discipline_comparison.png', width=1000)

## 6. Consistency vs Problems Relationship

Is there a relationship between cross-run consistency and within-run problems?

In [ ]:
Image('consistency_vs_problems.png', width=900)

In [ ]:
# Statistical analysis
from scipy import stats

# Correlation between consistency and within-run problems
correlation, p_value = stats.pearsonr(
    data['cross_run_consistency_pct'],
    data['within_run_duplicates'] + data['within_run_similar']
)

print(f"Correlation between cross-run consistency and within-run problems:")
print(f"  Pearson r = {correlation:.3f}")
print(f"  p-value = {p_value:.4f}")

if abs(correlation) < 0.3:
    print(f"  → Weak correlation: Cross-run consistency and within-run problems are largely independent")
elif abs(correlation) < 0.7:
    print(f"  → Moderate correlation")
else:
    print(f"  → Strong correlation")

# Compare consistency between topics with and without problems
with_problems = data[data['has_within_run_problems']]['cross_run_consistency_pct']
without_problems = data[~data['has_within_run_problems']]['cross_run_consistency_pct']

t_stat, t_p_value = stats.ttest_ind(with_problems, without_problems)

print(f"\nComparison of consistency (topics with vs without problems):")
print(f"  With problems: {with_problems.mean():.1f}% (±{with_problems.std():.1f}%)")
print(f"  Without problems: {without_problems.mean():.1f}% (±{without_problems.std():.1f}%)")
print(f"  t-test p-value: {t_p_value:.4f}")
if t_p_value < 0.05:
    print(f"  → Significant difference (p < 0.05)")
else:
    print(f"  → No significant difference (p ≥ 0.05)")

## 7. Interactive Dashboard

The interactive dashboard provides detailed hover information and drill-down capabilities.

In [ ]:
from IPython.display import IFrame
IFrame('prompt_consistency_dashboard.html', width=1000, height=1400)

## 8. Custom Analysis: Filter and Explore

Explore specific topics or patterns in detail.

In [ ]:
# Example: Find topics with high consistency AND no within-run problems (ideal!)
ideal_topics = data[
    (data['cross_run_consistency_pct'] > 20) & 
    (~data['has_within_run_problems'])
].sort_values('cross_run_consistency_pct', ascending=False)

print(f"Ideal Topics (High consistency, no within-run problems):")
display(ideal_topics[['discipline', 'topic', 'cross_run_consistency_pct', 'avg_questions_per_run']].head(10))

# Example: Find problematic topics (within-run problems AND low consistency)
problematic_topics = data[
    (data['has_within_run_problems']) & 
    (data['cross_run_consistency_pct'] < 10)
].sort_values('within_run_duplicates', ascending=False)

print(f"\nProblematic Topics (Within-run problems AND low consistency):")
display(problematic_topics[['discipline', 'topic', 'within_run_duplicates', 'cross_run_consistency_pct']].head(10))

In [ ]:
# Create a custom scatter plot with filtering
import plotly.express as px

# Add a category column for coloring
data['category'] = 'Normal'
data.loc[
    (data['cross_run_consistency_pct'] > 20) & (~data['has_within_run_problems']), 
    'category'
] = 'Ideal (High consistency, no problems)'
data.loc[
    (data['has_within_run_problems']) & (data['cross_run_consistency_pct'] < 10), 
    'category'
] = 'Problematic (Problems + low consistency)'
data.loc[
    data['has_within_run_problems'] & (data['cross_run_consistency_pct'] >= 10),
    'category'
] = 'Has within-run problems'

fig = px.scatter(
    data,
    x='cross_run_consistency_pct',
    y='within_run_duplicates',
    color='category',
    size='avg_questions_per_run',
    hover_name='topic',
    hover_data={
        'discipline': True,
        'cross_run_consistency_pct': ':.1f',
        'within_run_duplicates': True,
        'avg_questions_per_run': ':.1f',
        'category': False
    },
    title='Prompt Behavior Landscape: Consistency vs Within-Run Problems',
    labels={
        'cross_run_consistency_pct': 'Cross-Run Consistency (%)',
        'within_run_duplicates': 'Within-Run Duplicate Count'
    },
    color_discrete_map={
        'Ideal (High consistency, no problems)': '#388E3C',
        'Problematic (Problems + low consistency)': '#D32F2F',
        'Has within-run problems': '#F57C00',
        'Normal': '#1976D2'
    },
    width=1000,
    height=600
)

fig.update_layout(
    plot_bgcolor='white',
    font=dict(family='Arial', size=11, color='black')
)

fig.update_xaxes(showgrid=True, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridcolor='lightgray')

fig.show()

## 9. Export Data

Save the processed data for further analysis.

In [ ]:
# Save full dataset
data.to_csv('prompt_consistency_data.csv', index=False)
print("✓ Saved prompt_consistency_data.csv")

# Save problem topics only
problem_topics = data[data['has_within_run_problems']]
problem_topics.to_csv('topics_with_problems.csv', index=False)
print(f"✓ Saved topics_with_problems.csv ({len(problem_topics)} topics)")

# Save summary by discipline
summary.to_csv('discipline_summary.csv')
print("✓ Saved discipline_summary.csv")

## Summary

This analysis examined prompt consistency across 60 topics in three disciplines:

### Key Findings:

1. **Within-Run Problems** (Same prompt run produces duplicate questions):
   - Occurs in a subset of topics
   - Discrete Mathematics shows highest rate
   - Indicates the model sometimes repeats itself within a single generation

2. **Cross-Run Consistency** (Similarity across multiple runs of same prompt):
   - Moderate overall consistency (~16%)
   - Varies significantly by topic
   - Some topics produce very predictable questions, others vary greatly

3. **Relationship**:
   - Within-run problems and cross-run consistency appear to be largely independent
   - Topics can have high consistency without within-run problems (ideal)
   - Or have within-run problems with variable consistency (problematic)

### Recommendations:
- Investigate topics with within-run problems to improve prompts
- Consider whether high or low consistency is desirable for your use case
- Use topics without within-run problems for production question generation

### Accessibility Note:
All visualizations use WCAG AA-compliant, colorblind-safe color schemes with high contrast ratios.